In [26]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
import talib  # For Technical Indicators
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("processed_stock_data.csv", parse_dates=['Date'], index_col='Date')
df

,Open,High,Low,Close,Volume,50-day MA,200-day MA,Close_diff,lag_1,lag_7,rolling_mean_7,rolling_std_7
Date,,,,,,,,,,,,
2012-01-03,58.485714,58.928570,58.428570,58.747143,75555200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-01-04,58.571430,59.240002,58.468571,59.062859,65005500,NaN,NaN,0.315716,58.747143,NaN,NaN,NaN
2012-01-05,59.278572,59.792858,58.952858,59.718571,67817400,NaN,NaN,0.655712,59.062859,NaN,NaN,NaN
2012-01-06,59.967144,60.392857,59.888573,60.342857,79573200,NaN,NaN,0.624286,59.718571,NaN,NaN,NaN
2012-01-09,60.785713,61.107143,60.192856,60.247143,98506100,NaN,NaN,-0.095714,60.342857,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-23,280.529999,284.250000,280.369995,283.809289,24643000,259.670586,216.528297,4.369287,279.440002,271.459991,279.775608,2.524879
2019-12-24,284.150718,284.890015,280.977848,283.809289,12119700,260.629372,217.042793,0.000000,283.809289,275.149994,281.012650,1.932721
2019-12-26,284.150718,288.448204,280.977848,283.809289,23280300,261.599157,217.553289,0.000000,283.809289,279.859985,281.576836,2.108589


In [4]:
df = df.sort_index()

In [5]:
df.isnull().sum()

Open                0
High                0
Low                 0
Close               0
Volume              0
50-day MA          49
200-day MA        199
Close_diff          1
lag_1               1
lag_7               7
rolling_mean_7      6
rolling_std_7       6
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df

,Open,High,Low,Close,Volume,50-day MA,200-day MA,Close_diff,lag_1,lag_7,rolling_mean_7,rolling_std_7
Date,,,,,,,,,,,,
2012-01-03,58.485714,58.928570,58.428570,58.747143,75555200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2012-01-04,58.571430,59.240002,58.468571,59.062859,65005500,NaN,NaN,0.315716,58.747143,NaN,NaN,NaN
2012-01-05,59.278572,59.792858,58.952858,59.718571,67817400,NaN,NaN,0.655712,59.062859,NaN,NaN,NaN
2012-01-06,59.967144,60.392857,59.888573,60.342857,79573200,NaN,NaN,0.624286,59.718571,NaN,NaN,NaN
2012-01-09,60.785713,61.107143,60.192856,60.247143,98506100,NaN,NaN,-0.095714,60.342857,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-23,280.529999,284.250000,280.369995,283.809289,24643000,259.670586,216.528297,4.369287,279.440002,271.459991,279.775608,2.524879
2019-12-24,284.150718,284.890015,280.977848,283.809289,12119700,260.629372,217.042793,0.000000,283.809289,275.149994,281.012650,1.932721
2019-12-26,284.150718,288.448204,280.977848,283.809289,23280300,261.599157,217.553289,0.000000,283.809289,279.859985,281.576836,2.108589


In [9]:
df.drop(columns=["50-day MA", "200-day MA", "Close_diff", "lag_1", "lag_7", "rolling_mean_7", "rolling_std_7"], axis=1, inplace=True)
df

,Open,High,Low,Close,Volume
Date,,,,,
2012-01-03,58.485714,58.928570,58.428570,58.747143,75555200
2012-01-04,58.571430,59.240002,58.468571,59.062859,65005500
2012-01-05,59.278572,59.792858,58.952858,59.718571,67817400
2012-01-06,59.967144,60.392857,59.888573,60.342857,79573200
2012-01-09,60.785713,61.107143,60.192856,60.247143,98506100
...,...,...,...,...,...
2019-12-23,280.529999,284.250000,280.369995,283.809289,24643000
2019-12-24,284.150718,284.890015,280.977848,283.809289,12119700
2019-12-26,284.150718,288.448204,280.977848,283.809289,23280300


In [11]:
#Step 2: Feature Engineering
#Moving Averages (5-day, 10-day)
#Relative Strength Index (RSI)
#MACD (Moving Average Convergence Divergence)

In [12]:
# Simple Moving Averages (SMA)
df['SMA_5'] = df['Close'].rolling(window=5).mean()
df['SMA_10'] = df['Close'].rolling(window=10).mean()

In [14]:
def compute_rsi(data, window=14):
    delta = data.diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))


In [15]:
df['RSI'] = compute_rsi(df['Close'])

In [16]:
df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = df['EMA_12'] - df['EMA_26']
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

In [17]:
df.isnull().sum()

Open            0
High            0
Low             0
Close           0
Volume          0
SMA_5           4
SMA_10          9
RSI            13
EMA_12          0
EMA_26          0
MACD            0
MACD_Signal     0
dtype: int64

In [20]:
df.fillna(df.median(), inplace=True)

In [21]:
df.isnull().sum()

Open           0
High           0
Low            0
Close          0
Volume         0
SMA_5          0
SMA_10         0
RSI            0
EMA_12         0
EMA_26         0
MACD           0
MACD_Signal    0
dtype: int64

In [22]:
# Define features and target

In [23]:
features = ['Open', 'High', 'Low', 'Volume', 'SMA_5', 'SMA_10', 'RSI', 'MACD']
target = 'Close'

In [24]:
X = df[features]
y = df[target]

In [27]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [28]:
#Step 4: Train XGBoost Model


In [29]:
model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [30]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [32]:
#Step 5: Evaluate Model

In [33]:
# Predictions
y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"XGBoost MAE: {mae:.2f}")
print(f"XGBoost RMSE: {rmse:.2f}")

XGBoost MAE: 21.80
XGBoost RMSE: 32.71


In [34]:
!pip install optuna

  Using cached PyYAML-6.0.2-cp38-cp38-win_amd64.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.1 MB 3.3 MB/s eta 0:00:01
   --------- ------------------------------ 0.5/2.1 MB 3.3 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.1 MB 882.6 kB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 882.6 kB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 882.6 kB/s eta 0:00:02
   -------------- ------------------------- 0.8/2.1 MB 882.6 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 599.0 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 599.0 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 599.0 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 599.0 kB/s eta 0:00:02
   ------------------- -------------------- 1.0/2.1 MB 599.0 kB/s eta 0:00:02
   ----

In [35]:
import optuna

c:\Users\vimal\OneDrive\Desktop\Apple stock Forecast\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 0.5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10)
    }
    
    model = xgb.XGBRegressor(**params, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    return mean_absolute_error(y_test, y_pred)


In [38]:
#Step 3: Run Optuna to Find the Best Hyperparameters

In [41]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=100) 

[I 2025-03-01 13:55:52,708] A new study created in memory with name: no-name-4ae5e24a-6ed2-43d5-9bd7-3b7014a3b6b4
[I 2025-03-01 13:55:53,171] Trial 0 finished with value: 21.817623471885657 and parameters: {'n_estimators': 250, 'learning_rate': 0.26282019851502536, 'max_depth': 9, 'subsample': 0.8710420825718012, 'colsample_bytree': 0.6227826360720016, 'gamma': 0.48098996477493616, 'reg_alpha': 0.8347045054110314, 'reg_lambda': 5.781738746230866}. Best is trial 0 with value: 21.817623471885657.
[I 2025-03-01 13:55:53,663] Trial 1 finished with value: 22.317556492306235 and parameters: {'n_estimators': 400, 'learning_rate': 0.06666880407834208, 'max_depth': 10, 'subsample': 0.9764605845592338, 'colsample_bytree': 0.6042968760469473, 'gamma': 0.25442885845715085, 'reg_alpha': 5.27244983376233, 'reg_lambda': 3.7023405321557545}. Best is trial 0 with value: 21.817623471885657.
[I 2025-03-01 13:55:54,369] Trial 2 finished with value: 22.695959313084 and parameters: {'n_estimators': 450, 'le

In [40]:
#Train the Final Model with Best Hyperparameters

In [44]:
best_params = study.best_params
best_params

{'n_estimators': 500,
 'learning_rate': 0.07023534506844975,
 'max_depth': 8,
 'subsample': 0.9415256107815246,
 'colsample_bytree': 0.9611898445067141,
 'gamma': 0.44111045385377884,
 'reg_alpha': 0.04100512352871791,
 'reg_lambda': 1.5459592402535849}

In [43]:
model = xgb.XGBRegressor(**best_params, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"Tuned XGBoost MAE: {mae:.2f}")
print(f"Tuned XGBoost RMSE: {rmse:.2f}")
print("Best Parameters:", best_params)

Tuned XGBoost MAE: 21.55
Tuned XGBoost RMSE: 32.51
Best Parameters: {'n_estimators': 500, 'learning_rate': 0.07023534506844975, 'max_depth': 8, 'subsample': 0.9415256107815246, 'colsample_bytree': 0.9611898445067141, 'gamma': 0.44111045385377884, 'reg_alpha': 0.04100512352871791, 'reg_lambda': 1.5459592402535849}


In [38]:
def create_features(data, lags=20):
    df = data.copy()
    for lag in range(1, lags + 1):
        df[f'lag_{lag}'] = df[target_column].shift(lag)
    
    # Moving Averages
    df['SMA_5'] = df[target_column].rolling(window=5).mean()
    df['SMA_10'] = df[target_column].rolling(window=10).mean()
    
    df.dropna(inplace=True)
    return df

In [39]:
df_lagged = create_features(df, lags=20)

In [40]:
df_lagged

,Open,High,Low,Close,Volume,50-day MA,200-day MA,Close_diff,lag_1,lag_7,...,lag_13,lag_14,lag_15,lag_16,lag_17,lag_18,lag_19,lag_20,SMA_5,SMA_10
Date,,,,,,,,,,,,,,,,,,,,,
2012-10-16,90.767143,92.900002,90.142860,92.827141,137442900,94.422257,82.828500,2.147141,90.680000,93.227142,...,97.331429,95.025711,96.220001,98.684288,100.012856,99.814285,100.300003,100.272858,90.950571,92.116142
2012-10-17,92.695717,93.255714,92.000000,92.087143,97259400,94.489971,82.995200,-0.739998,92.827141,91.167145,...,95.300003,97.331429,95.025711,96.220001,98.684288,100.012856,99.814285,100.300003,91.056285,91.732714
2012-10-18,91.370003,91.722855,90.000000,90.377144,119156100,94.526486,83.151771,-1.709999,92.087143,90.835716,...,94.198570,95.300003,97.331429,95.025711,96.220001,98.684288,100.012856,99.814285,91.186000,91.244714
2012-10-19,90.150002,90.252853,87.088570,87.120003,145397275,94.495372,83.288779,-3.257141,90.377144,91.558571,...,94.472855,94.198570,95.300003,97.331429,95.025711,96.220001,98.684288,100.012856,90.618286,90.634000
2012-10-22,87.488571,90.768570,87.251427,90.575714,136682700,94.530600,83.439943,3.455711,87.120003,89.728569,...,95.921425,94.472855,94.198570,95.300003,97.331429,95.025711,96.220001,98.684288,90.597429,90.574857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-23,280.529999,284.250000,280.369995,283.809289,24643000,259.670586,216.528297,4.369287,279.440002,271.459991,...,261.739990,259.450012,264.160004,267.250000,267.839996,264.290009,266.369995,261.779999,280.683855,276.913924
2019-12-24,284.150718,284.890015,280.977848,283.809289,12119700,260.629372,217.042793,0.000000,283.809289,275.149994,...,265.579987,261.739990,259.450012,264.160004,267.250000,267.839996,264.290009,266.369995,281.363712,278.446852
2019-12-26,284.150718,288.448204,280.977848,283.809289,23280300,261.599157,217.553289,0.000000,283.809289,279.859985,...,270.709991,265.579987,261.739990,259.450012,264.160004,267.250000,267.839996,264.290009,282.177572,279.750782


In [41]:
train_size = int(len(df_lagged) * 0.8)
train, test = df_lagged[:train_size], df_lagged[train_size:]

In [42]:
X_train, y_train = train.drop(columns=[target_column]), train[target_column]
X_test, y_test = test.drop(columns=[target_column]), test[target_column]

In [43]:
train_dmatrix = xgb.DMatrix(X_train, label=y_train)
test_dmatrix = xgb.DMatrix(X_test, label=y_test)

In [44]:
param_grid = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'learning_rate': 0.03,
    'max_depth': 6,
    'min_child_weight': 5,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'n_estimators': 1000,
    'early_stopping_rounds': 20
}

In [45]:
xgb_model = xgb.train(
    params=param_grid,
    dtrain=train_dmatrix,
    num_boost_round=1000,
    evals=[(train_dmatrix, 'train'), (test_dmatrix, 'test')],
    early_stopping_rounds=20,
    verbose_eval=50
)

[0]	train-rmse:34.30542	test-rmse:96.46211
[50]	train-rmse:7.73393	test-rmse:46.39983


c:\Users\vimal\OneDrive\Desktop\Apple stock Forecast\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [01:51:32] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "early_stopping_rounds", "n_estimators" } are not used.

  warnings.warn(smsg, UserWarning)


[100]	train-rmse:1.82129	test-rmse:35.14980
[150]	train-rmse:0.53445	test-rmse:32.38172
[200]	train-rmse:0.28942	test-rmse:31.59870
[250]	train-rmse:0.23660	test-rmse:31.27065
[300]	train-rmse:0.21361	test-rmse:31.19903
[318]	train-rmse:0.20789	test-rmse:31.20934


In [48]:
y_pred = xgb_model.predict(test_dmatrix)

In [49]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"🔍 Improved XGBoost Model:\nRMSE: {rmse:.2f}\nMAE: {mae:.2f}")

🔍 Improved XGBoost Model:
RMSE: 31.21
MAE: 20.34


In [4]:
#2. Feature Engineering
#Create Lag Features (previous time step values as features).
#Create Rolling Statistics (e.g., moving averages).
#Extract Date Features (year, month, day, weekday).

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2011 entries, 0 to 2010
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Date            2011 non-null   object 
 1   Open            2011 non-null   float64
 2   High            2011 non-null   float64
 3   Low             2011 non-null   float64
 4   Close           2011 non-null   float64
 5   Volume          2011 non-null   int64  
 6   50-day MA       1962 non-null   float64
 7   200-day MA      1812 non-null   float64
 8   Close_diff      2010 non-null   float64
 9   lag_1           2010 non-null   float64
 10  lag_7           2004 non-null   float64
 11  rolling_mean_7  2005 non-null   float64
 12  rolling_std_7   2005 non-null   float64
dtypes: float64(11), int64(1), object(1)
memory usage: 204.4+ KB


In [7]:
df['Date'] = pd.to_datetime(df['Date'])  # Convert to datetime
df.set_index('Date', inplace=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2011 entries, 2012-01-03 to 2019-12-30
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Open            2011 non-null   float64
 1   High            2011 non-null   float64
 2   Low             2011 non-null   float64
 3   Close           2011 non-null   float64
 4   Volume          2011 non-null   int64  
 5   50-day MA       1962 non-null   float64
 6   200-day MA      1812 non-null   float64
 7   Close_diff      2010 non-null   float64
 8   lag_1           2010 non-null   float64
 9   lag_7           2004 non-null   float64
 10  rolling_mean_7  2005 non-null   float64
 11  rolling_std_7   2005 non-null   float64
dtypes: float64(11), int64(1)
memory usage: 204.2 KB


In [9]:
#3. Train-Test Split

In [10]:
train_size = int(len(df) * 0.8)  # 80% training, 20% testing
train, test = df.iloc[:train_size], df.iloc[train_size:]

In [11]:
# Features (X) and Target (y)
X_train, y_train = train.drop(columns=['Close']), train['Close']
X_test, y_test = test.drop(columns=['Close']), test['Close']

In [ ]:
#Train XGBoost Model

In [13]:
from xgboost import XGBRegressor

In [14]:
model = XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1)

In [15]:
model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [16]:
y_pred = model.predict(X_test)

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [18]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

In [19]:
print(f'RMSE: {rmse:.2f}')
print(f'MAE: {mae:.2f}')

RMSE: 32.20
MAE: 21.17
